# RetinaSafe — Kermany OCT2017 baseline (ResNet-50)

**Goal:** train a ResNet-50 on the Kermany OCT2017 dataset under a fresh **patient-disjoint** split, with class-weighted loss to handle imbalance, and report calibrated test metrics with bootstrap 95% CIs.

**Status of decisions already validated:**
- Dataset attached at `/kaggle/input/datasets/paultimothymooney/kermany2018/OCT2017 ` (trailing space deliberate).
- 84,484 images, 4 classes (NORMAL/CNV/DME/DRUSEN), 4,657 unique patients.
- Released splits leak 566 patients across train↔test → we resplit patient-disjointly (70/15/15, stratified by modal class, seed 42).
- Images are grayscale, mixed sizes (768×496, 512×512, 512×496) → resize to 224 + RGB-replicate.

**How to run:**

1. Open Kaggle, this notebook, attach the **kermany2018** dataset.
2. Settings → Internet **On** (Cell 6 downloads ImageNet weights once).
3. Run cells 1–6 interactively (CPU is fine) to confirm setup.
4. Settings → Accelerator → **GPU T4 x2**.
5. File → Save Version → **Save & Run All (Commit)** to run training headless.

**Expected runtime:** ~3–5 hours on a T4.

## Cell 1 — Imports + GPU + path check

In [ ]:
import os, sys, json, time, random, pathlib
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
from PIL import Image
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix

# ---------- Reproducibility ----------
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

# ---------- Paths ----------
DATA_ROOT = "/kaggle/input/datasets/paultimothymooney/kermany2018/OCT2017 "
WORK = pathlib.Path("/kaggle/working")
INDEX_CSV = WORK / "kermany_index.csv"
RESULTS = WORK / "results"
RESULTS.mkdir(exist_ok=True)
CKPT = RESULTS / "best.pt"

# ---------- Environment ----------
print("Torch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert os.path.exists(DATA_ROOT), f"Dataset not attached. Add kermany2018 to the notebook."
print("Data root OK:", DATA_ROOT)

## Cell 2 — Build patient-disjoint index

Walks the dataset once, extracts patient IDs from filenames (`<CLASS>-<PATIENT_ID>-<FRAME>.jpeg`), and assigns each *patient* to exactly one of train/val/test. Stratified by each patient's modal class. Saves to `kermany_index.csv`.

**Idempotent:** if the CSV already exists, skip rebuilding.

In [ ]:
if INDEX_CSV.exists():
    df = pd.read_csv(INDEX_CSV)
    print(f"Index already exists at {INDEX_CSV}  ({len(df):,} rows) — skipping rebuild.")
else:
    rows = []
    for split in ["train", "val", "test"]:
        for cls in ["CNV", "DME", "DRUSEN", "NORMAL"]:
            cls_dir = os.path.join(DATA_ROOT, split, cls)
            for fname in os.listdir(cls_dir):
                if not fname.lower().endswith((".jpeg", ".jpg", ".png")):
                    continue
                patient_id = fname.split("-")[1]
                rows.append({
                    "path": os.path.join(cls_dir, fname),
                    "native_class": cls,
                    "patient_id": patient_id,
                })
    df = pd.DataFrame(rows)

    # Modal-class stratification: each patient assigned to ONE split
    pat2cls = df.groupby("patient_id")["native_class"].agg(lambda s: Counter(s).most_common(1)[0][0])
    rng = np.random.default_rng(SEED)
    TRAIN_FRAC, VAL_FRAC = 0.70, 0.15
    pat2split = {}
    for cls in pat2cls.unique():
        patients = sorted(pat2cls[pat2cls == cls].index.tolist())
        rng.shuffle(patients)
        n = len(patients)
        n_train, n_val = int(round(TRAIN_FRAC * n)), int(round(VAL_FRAC * n))
        for i, pid in enumerate(patients):
            if i < n_train:           pat2split[pid] = "train"
            elif i < n_train + n_val: pat2split[pid] = "val"
            else:                     pat2split[pid] = "test"
    df["split"] = df["patient_id"].map(pat2split)

    # Verify no leakage
    bad = df.groupby("patient_id")["split"].nunique()
    assert (bad <= 1).all(), f"Resplit leaks {int((bad>1).sum())} patients."

    df[["path", "split", "native_class", "patient_id"]].to_csv(INDEX_CSV, index=False)
    print(f"Wrote {INDEX_CSV} ({len(df):,} rows)")

print("\nClass distribution by split:")
print(df.groupby(["split", "native_class"]).size().unstack(fill_value=0))
print("\nPatient count by split:")
print(df.groupby("split")["patient_id"].nunique())

## Cell 3 — Visualize a few samples

In [ ]:
import matplotlib.pyplot as plt

samples = (df[df['split'] == 'train']
           .groupby('native_class', group_keys=False)
           .sample(n=2, random_state=SEED)
           .reset_index(drop=True))

fig, axes = plt.subplots(4, 2, figsize=(10, 14))
for i, row in samples.iterrows():
    img = Image.open(row['path'])
    ax = axes[i // 2, i % 2]
    ax.imshow(img, cmap='gray')
    ax.set_title(f"{row['native_class']}  |  patient {row['patient_id']}  |  {img.size}")
    ax.axis('off')
plt.tight_layout(); plt.show()

## Cell 4 — Dataset class + transforms

Train transform: light augmentation (random crop after slight upscale, horizontal flip, mild color jitter). Eval transform: deterministic resize.

In [ ]:
CLASSES = ["NORMAL", "CNV", "DME", "DRUSEN"]
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
N_CLASSES = len(CLASSES)
IMG_SIZE = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]


def train_tf(size=IMG_SIZE):
    return transforms.Compose([
        transforms.Grayscale(num_output_channels=3),
        transforms.Resize((size + 32, size + 32)),
        transforms.RandomCrop(size),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ColorJitter(brightness=0.1, contrast=0.1),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])

def eval_tf(size=IMG_SIZE):
    return transforms.Compose([
        transforms.Grayscale(num_output_channels=3),
        transforms.Resize((size, size)),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])


class KermanyDataset(Dataset):
    def __init__(self, index_csv, split, transform=None):
        full = pd.read_csv(index_csv)
        self.df = full[full["split"] == split].reset_index(drop=True)
        self.transform = transform
        self.labels = self.df["native_class"].map(CLASS_TO_IDX).to_numpy()

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["path"])
        if self.transform:
            img = self.transform(img)
        return img, int(self.labels[idx]), row["patient_id"]


print("Dataset class + transforms defined.")

## Cell 5 — Build loaders + sanity-check one batch

Class-balanced sampling for the train loader (CNV is ~4× DRUSEN, would dominate otherwise).

In [ ]:
BATCH_SIZE = 64
NUM_WORKERS = 2

train_ds = KermanyDataset(INDEX_CSV, "train", transform=train_tf())
val_ds   = KermanyDataset(INDEX_CSV, "val",   transform=eval_tf())
test_ds  = KermanyDataset(INDEX_CSV, "test",  transform=eval_tf())
print(f"train={len(train_ds):,}  val={len(val_ds):,}  test={len(test_ds):,}")

# Class-balanced sampler for training
class_counts = np.bincount(train_ds.labels, minlength=N_CLASSES)
sample_weights = 1.0 / class_counts[train_ds.labels]
sampler = WeightedRandomSampler(
    weights=torch.DoubleTensor(sample_weights),
    num_samples=len(train_ds),
    replacement=True,
)
print(f"Class counts (train): {dict(zip(CLASSES, class_counts.tolist()))}")

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

# Sanity: pull one training batch
imgs, labels, pids = next(iter(train_loader))
print(f"\nBatch shape: {imgs.shape}  dtype={imgs.dtype}  range=[{imgs.min():.2f}, {imgs.max():.2f}]")
print(f"Batch class distribution (sampler should roughly balance):")
for cls_idx, n in Counter(labels.tolist()).items():
    print(f"  {CLASSES[cls_idx]:7s}: {n}")

## Cell 6 — Build the ResNet-50 model

ImageNet-pretrained backbone, 4-class head, light dropout. First run downloads weights from torchvision (requires Internet ON).

In [ ]:
def build_resnet50(num_classes=N_CLASSES, dropout=0.2):
    weights = models.ResNet50_Weights.IMAGENET1K_V2
    m = models.resnet50(weights=weights)
    in_feat = m.fc.in_features
    m.fc = nn.Sequential(
        nn.Dropout(dropout),
        nn.Linear(in_feat, num_classes),
    )
    return m

model = build_resnet50().to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"ResNet-50 built: {n_params/1e6:.2f}M params")

# Quick forward-pass sanity check on the batch from Cell 5
model.eval()
with torch.no_grad():
    out = model(imgs.to(DEVICE))
print(f"Forward pass OK. Output shape: {out.shape}  (expected: [{BATCH_SIZE}, {N_CLASSES}])")

## Cell 7 — Training loop

**Setup:** AdamW (lr 1e-4, wd 1e-4), cosine LR schedule with 2-epoch warmup, AMP (mixed precision), early stopping on val macro-AUROC with patience 6, max 30 epochs.

**Loss:** class-weighted cross-entropy. We already balance via the WeightedRandomSampler, but adding class weights gives a small additional bump for the rare classes.

**Logs:** per-epoch train/val loss + val macro-AUROC. Best checkpoint saved to `/kaggle/working/results/best.pt`.

In [ ]:
EPOCHS = 30
LR = 1e-4
WD = 1e-4
WARMUP_EPOCHS = 2
PATIENCE = 6

@torch.no_grad()
def evaluate(model, loader, device, return_logits=False):
    model.eval()
    losses, all_logits, all_labels = [], [], []
    crit = nn.CrossEntropyLoss()
    for x, y, _ in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        with torch.cuda.amp.autocast():
            logits = model(x)
            loss = crit(logits, y)
        losses.append(loss.item() * x.size(0))
        all_logits.append(logits.float().cpu())
        all_labels.append(y.cpu())
    n = sum(t.size(0) for t in all_labels)
    avg_loss = sum(losses) / n
    logits = torch.cat(all_logits)
    labels = torch.cat(all_labels)
    probs = F.softmax(logits, dim=1).numpy()
    y_true = labels.numpy()
    # Macro AUROC (one-vs-rest) on classes that appear in the split
    present = sorted(np.unique(y_true).tolist())
    aurocs = []
    for c in present:
        y_bin = (y_true == c).astype(int)
        if y_bin.sum() == 0 or y_bin.sum() == len(y_bin):
            continue
        aurocs.append(roc_auc_score(y_bin, probs[:, c]))
    macro_auroc = float(np.mean(aurocs)) if aurocs else float("nan")
    acc = accuracy_score(y_true, probs.argmax(axis=1))
    if return_logits:
        return avg_loss, macro_auroc, acc, logits, labels
    return avg_loss, macro_auroc, acc


def cosine_warmup_lr(epoch, total_epochs, warmup, base_lr):
    if epoch < warmup:
        return base_lr * (epoch + 1) / warmup
    progress = (epoch - warmup) / max(1, total_epochs - warmup)
    return base_lr * 0.5 * (1 + np.cos(np.pi * progress))


# Class-weighted CE (inverse frequency, normalized)
cw = torch.tensor(1.0 / class_counts, dtype=torch.float32)
cw = cw / cw.mean()
criterion = nn.CrossEntropyLoss(weight=cw.to(DEVICE))

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

best_auroc = -1.0
best_epoch = -1
no_improve = 0
history = []

for epoch in range(EPOCHS):
    lr_this = cosine_warmup_lr(epoch, EPOCHS, WARMUP_EPOCHS, LR)
    for g in optimizer.param_groups:
        g["lr"] = lr_this

    model.train()
    t0 = time.time()
    running = 0.0
    n_seen = 0
    for x, y, _ in train_loader:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            logits = model(x)
            loss = criterion(logits, y)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running += loss.item() * x.size(0)
        n_seen += x.size(0)
    train_loss = running / n_seen

    val_loss, val_auroc, val_acc = evaluate(model, val_loader, DEVICE)
    dur = time.time() - t0
    history.append({"epoch": epoch, "lr": lr_this, "train_loss": train_loss,
                    "val_loss": val_loss, "val_auroc": val_auroc, "val_acc": val_acc,
                    "seconds": dur})
    print(f"Epoch {epoch+1:02d}/{EPOCHS}  lr={lr_this:.2e}  "
          f"train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  "
          f"val_auroc={val_auroc:.4f}  val_acc={val_acc:.4f}  ({dur:.0f}s)")

    if val_auroc > best_auroc:
        best_auroc = val_auroc
        best_epoch = epoch
        no_improve = 0
        torch.save({
            "model_state": model.state_dict(),
            "epoch": epoch,
            "val_auroc": val_auroc,
            "classes": CLASSES,
        }, CKPT)
        print(f"  -> new best (val_auroc={val_auroc:.4f}), saved {CKPT}")
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print(f"Early stopping at epoch {epoch+1} (no improvement for {PATIENCE} epochs)")
            break

pd.DataFrame(history).to_csv(RESULTS / "training_history.csv", index=False)
print(f"\nBest val_auroc={best_auroc:.4f} at epoch {best_epoch+1}")

## Cell 8 — Test eval with bootstrap 95% CIs

Loads the best checkpoint, runs on the held-out test set, computes per-class AUROC + macro AUROC + accuracy with 1000-resample bootstrap confidence intervals.

In [ ]:
state = torch.load(CKPT, map_location=DEVICE)
model.load_state_dict(state["model_state"])
print(f"Loaded best checkpoint from epoch {state['epoch']+1} (val_auroc={state['val_auroc']:.4f})")

test_loss, test_auroc_macro, test_acc, test_logits, test_labels = evaluate(
    model, test_loader, DEVICE, return_logits=True
)
test_probs = F.softmax(test_logits, dim=1).numpy()
y_true = test_labels.numpy()

# Per-class AUROC (one-vs-rest)
per_class_auroc = {}
for i, c in enumerate(CLASSES):
    y_bin = (y_true == i).astype(int)
    if y_bin.sum() == 0:
        per_class_auroc[c] = None
    else:
        per_class_auroc[c] = float(roc_auc_score(y_bin, test_probs[:, i]))

# Confusion matrix
preds = test_probs.argmax(axis=1)
cm = confusion_matrix(y_true, preds, labels=list(range(N_CLASSES)))

# Bootstrap CI
def bootstrap_ci(probs, y_true, n_boot=1000, alpha=0.05, seed=SEED):
    rng = np.random.default_rng(seed)
    n = len(y_true)
    macro_aurocs, accs = [], []
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        pb, yb = probs[idx], y_true[idx]
        # Macro AUROC over classes present in this resample
        aurocs = []
        for c in range(N_CLASSES):
            y_bin = (yb == c).astype(int)
            if y_bin.sum() == 0 or y_bin.sum() == len(y_bin):
                continue
            aurocs.append(roc_auc_score(y_bin, pb[:, c]))
        if aurocs:
            macro_aurocs.append(np.mean(aurocs))
        accs.append(accuracy_score(yb, pb.argmax(axis=1)))
    def ci(v):
        v = np.asarray(v)
        return [float(np.quantile(v, alpha/2)), float(np.quantile(v, 1-alpha/2))]
    return ci(macro_aurocs), ci(accs)

auroc_ci, acc_ci = bootstrap_ci(test_probs, y_true)

results = {
    "test_n": int(len(y_true)),
    "test_loss": float(test_loss),
    "test_accuracy": float(test_acc),
    "test_accuracy_ci95": acc_ci,
    "test_auroc_macro": float(test_auroc_macro),
    "test_auroc_macro_ci95": auroc_ci,
    "test_auroc_per_class": per_class_auroc,
    "confusion_matrix": cm.tolist(),
    "classes": CLASSES,
    "best_epoch": int(state["epoch"] + 1),
    "best_val_auroc": float(state["val_auroc"]),
}

with open(RESULTS / "test_metrics.json", "w") as f:
    json.dump(results, f, indent=2)

print("\n=== Test results ===")
print(json.dumps(results, indent=2))

print("\nConfusion matrix (rows = true, cols = pred):")
print("        " + "  ".join(f"{c:>6s}" for c in CLASSES))
for i, c in enumerate(CLASSES):
    print(f"  {c:>6s}  " + "  ".join(f"{cm[i,j]:>6d}" for j in range(N_CLASSES)))

## Cell 9 — Persist artifacts off Kaggle

Zips the results directory (best.pt + test_metrics.json + training_history.csv) into a single download. Available from the notebook's Output tab after Commit completes.

In [ ]:
import shutil
zip_path = WORK / "kermany_baseline_results.zip"
if zip_path.exists():
    zip_path.unlink()
shutil.make_archive(zip_path.with_suffix(""), "zip", root_dir=str(RESULTS))
print(f"Wrote {zip_path}  ({zip_path.stat().st_size/1e6:.1f} MB)")
print()
print("Download from: Notebook page → Output tab → kermany_baseline_results.zip")